# 01 - Synthetic Dataset Generation

Generates a synthetic function-calling dataset for fine-tuning Mistral 7B.

**Approach**: Use Claude via the Databricks Foundation Model API to generate 
diverse user queries paired with correct tool call responses across 16 enterprise 
tool schemas and 5 difficulty categories. Validates against tool schemas, and saves train/val/test splits.

**Output**: `train.jsonl`, `val.jsonl`, `test.jsonl`

In [0]:
import sys
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Add project root to path so we can import src/
# Adjust this path to wherever your repo is cloned/mounted in Databricks
PROJECT_ROOT = "/Workspace/Users/alberto.lapedriza@kpmg.co.uk/learning/mistral-7b-enterprise-function-calling"
sys.path.insert(0, PROJECT_ROOT)

from src.schemas import TOOL_SCHEMAS, TOOL_NAMES, SYSTEM_PROMPT
from src.generation import build_generation_plan, run_generation
from src.validation import parse_model_response
from src.utils import (
    to_training_format, deduplicate_examples,
    save_jsonl, load_jsonl, spot_check, dataset_stats
)

In [0]:
from openai import OpenAI

DATABRICKS_HOST = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().apiUrl().getOrElse(None)
)
DATABRICKS_TOKEN = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().apiToken().getOrElse(None)
)

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"{DATABRICKS_HOST}/serving-endpoints"
)

CLAUDE_MODEL = "databricks-claude-sonnet-4-6"  # <-- UPDATE to match your endpoint

def call_claude(prompt: str) -> str:
    """Call Claude Opus via Databricks."""
    response = client.chat.completions.create(
        model=CLAUDE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=4096,
        temperature=0.8,
    )
    return response.choices[0].message.content

# Quick test
print(call_claude("Say 'hello' and nothing else."))

hello


In [0]:
plan = build_generation_plan()
print(f"Total generation tasks: {len(plan)}")

summary = pd.DataFrame([
    {"category": t.category, "expected": t.expected_count} for t in plan
])
print(summary.groupby("category")["expected"].sum())
print(f"\nTotal expected: {summary['expected'].sum()}")

Total generation tasks: 178
category
ambiguous     160
complex       384
multi_tool    240
no_tool       160
simple        640
Name: expected, dtype: int64

Total expected: 1584


In [0]:
raw_examples, gen_log = run_generation(
    tasks=plan,
    llm_call_fn=call_claude,
    max_retries=2,
    delay_between_calls=1.5,
)

print(f"\n✅ Generated {len(raw_examples)} valid examples")

  ✅ [1/178] simple | get_transaction_history | attempt 1 | 10/10 valid
  ✅ [2/178] simple | get_transaction_history | attempt 1 | 10/10 valid
  ✅ [3/178] simple | get_transaction_history | attempt 1 | 10/10 valid
  ✅ [4/178] simple | get_transaction_history | attempt 1 | 10/10 valid
  ✅ [5/178] simple | create_invoice | attempt 1 | 10/10 valid
  ✅ [6/178] simple | create_invoice | attempt 1 | 10/10 valid
  ✅ [7/178] simple | create_invoice | attempt 1 | 10/10 valid
  ✅ [8/178] simple | create_invoice | attempt 1 | 10/10 valid
  ✅ [9/178] simple | run_compliance_check | attempt 1 | 10/10 valid
  ✅ [10/178] simple | run_compliance_check | attempt 1 | 10/10 valid
  ✅ [11/178] simple | run_compliance_check | attempt 1 | 10/10 valid
  ✅ [12/178] simple | run_compliance_check | attempt 1 | 10/10 valid
  ✅ [13/178] simple | get_portfolio_summary | attempt 1 | 10/10 valid
  ✅ [14/178] simple | get_portfolio_summary | attempt 1 | 10/10 valid
  ✅ [15/178] simple | get_portfolio_summary | attempt

  ✅ [45/178] simple | send_notification | attempt 1 | 10/10 valid
  ✅ [46/178] simple | send_notification | attempt 1 | 10/10 valid
  ✅ [47/178] simple | send_notification | attempt 1 | 10/10 valid
  ✅ [48/178] simple | send_notification | attempt 1 | 10/10 valid
  ✅ [49/178] simple | create_audit_task | attempt 1 | 10/10 valid
  ✅ [50/178] simple | create_audit_task | attempt 1 | 10/10 valid
  ✅ [51/178] simple | create_audit_task | attempt 1 | 10/10 valid
  ✅ [52/178] simple | create_audit_task | attempt 1 | 10/10 valid
  ✅ [53/178] simple | escalate_issue | attempt 1 | 10/10 valid
  ✅ [54/178] simple | escalate_issue | attempt 1 | 10/10 valid
  ✅ [55/178] simple | escalate_issue | attempt 1 | 10/10 valid
  ✅ [56/178] simple | escalate_issue | attempt 1 | 10/10 valid
  ✅ [57/178] simple | get_workflow_status | attempt 1 | 10/10 valid
  ✅ [58/178] simple | get_workflow_status | attempt 1 | 10/10 valid
  ✅ [59/178] simple | get_workflow_status | attempt 1 | 10/10 valid
  ✅ [60/178] sim

  ✅ [103/178] complex | create_audit_task | attempt 1 | 8/8 valid
  ✅ [104/178] complex | escalate_issue | attempt 1 | 8/8 valid
  ✅ [105/178] complex | escalate_issue | attempt 1 | 8/8 valid
  ✅ [106/178] complex | escalate_issue | attempt 1 | 8/8 valid
  ✅ [107/178] complex | get_workflow_status | attempt 1 | 8/8 valid
  ✅ [108/178] complex | get_workflow_status | attempt 1 | 8/8 valid
  ✅ [109/178] complex | get_workflow_status | attempt 1 | 8/8 valid
  ✅ [110/178] complex | log_time_entry | attempt 1 | 8/8 valid
  ✅ [111/178] complex | log_time_entry | attempt 1 | 8/8 valid
  ✅ [112/178] complex | log_time_entry | attempt 1 | 8/8 valid
  ✅ [113/178] multi_tool | search_customers + get_customer_risk_profile | attempt 1 | 8/8 valid
  ✅ [114/178] multi_tool | search_customers + get_customer_risk_profile | attempt 1 | 8/8 valid
  ✅ [115/178] multi_tool | search_customers + get_customer_risk_profile | attempt 1 | 8/8 valid
  ✅ [116/178] multi_tool | run_compliance_check + get_customer_r

In [0]:
print(f"Successful: {len(gen_log[gen_log['status'] == 'success'])}")
print(f"Failed: {len(gen_log[gen_log['status'] != 'success'])}")
print(f"\nValid by category:\n{gen_log.groupby('category')['valid'].sum()}")
print(f"\nInvalid by category:\n{gen_log.groupby('category')['invalid'].sum()}")

failed = gen_log[gen_log["status"] != "success"]
if len(failed) > 0:
    print(f"\n⚠️ Failed tasks:\n{failed[['task_id', 'status']]}")

Successful: 178
Failed: 0

Valid by category:
category
ambiguous     160
complex       384
multi_tool    240
no_tool       160
simple        640
Name: valid, dtype: int64

Invalid by category:
category
ambiguous     0
complex       0
multi_tool    0
no_tool       0
simple        0
Name: invalid, dtype: int64


In [0]:
training_examples = [to_training_format(ex) for ex in raw_examples]
training_examples = deduplicate_examples(training_examples)

Removed 122 duplicates | Remaining: 1462


In [0]:
# Extract categories for stratification
categories = [ex.get("category", ex.get("_category", "unknown")) for ex in training_examples]

# First split: 80% train, 20% temp (val + test)
train_set, temp_set, train_cats, temp_cats = train_test_split(
    training_examples, categories, test_size=0.2, stratify=categories, random_state=42
)

# Second split: split temp 50/50 into val (10%) and test (10%)
val_set, test_set = train_test_split(
    temp_set, test_size=0.5, stratify=temp_cats, random_state=42
)

print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

Train: 1169 | Val: 146 | Test: 147


In [0]:
for cat in ["simple", "complex", "multi_tool", "ambiguous", "no_tool"]:
    print(f"\n{'#'*70}")
    print(f"# {cat.upper()}")
    print(f"{'#'*70}")
    spot_check(training_examples, n=3, category=cat)


######################################################################
# SIMPLE
######################################################################

Example 1 | Category: simple
USER: Hey, can you kick off a KYC check for Orion Global Fund? It's a fund and the jurisdiction is US. Make it a full depth check please.

ASSISTANT: {"name": "run_compliance_check", "arguments": {"entity_name": "Orion Global Fund", "entity_type": "fund", "regulation": "KYC", "jurisdiction": "US", "check_depth": "full"}}

Example 2 | Category: simple
USER: What's the deal with workflow IMPL-9920? Just need a quick status.

ASSISTANT: {"name": "get_workflow_status", "arguments": {"workflow_id": "IMPL-9920"}}

Example 3 | Category: simple
USER: Quick time log: ENG-73904, 2024-10-14, 7 hours, fieldwork. Spent the full day at Meridian Logistics testing accounts payable transactions and vouching invoices.

ASSISTANT: {"name": "log_time_entry", "arguments": {"engagement_id": "ENG-73904", "date": "2024-10-14", "ho

In [0]:
OUTPUT_DIR = f"{PROJECT_ROOT}/data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

save_jsonl(train_set, f"{OUTPUT_DIR}/train.jsonl", strip_category=True)
save_jsonl(val_set, f"{OUTPUT_DIR}/val.jsonl", strip_category=True)
save_jsonl(test_set, f"{OUTPUT_DIR}/test.jsonl", strip_category=False)  # keep category for eval

Saved 1169 examples → /Workspace/Users/alberto.lapedriza@kpmg.co.uk/learning/mistral-7b-enterprise-function-calling/data/train.jsonl
Saved 146 examples → /Workspace/Users/alberto.lapedriza@kpmg.co.uk/learning/mistral-7b-enterprise-function-calling/data/val.jsonl
Saved 147 examples → /Workspace/Users/alberto.lapedriza@kpmg.co.uk/learning/mistral-7b-enterprise-function-calling/data/test.jsonl
